In [ ]:
"""


Usage: python age_difference_stats_minimal.py [path_to_xlsx]
Requires: pandas, numpy, scipy, openpyxl
"""

import os
import sys

import numpy as np
import pandas as pd
from scipy import stats

DATA_PATH = r""
AGE_COL = "Age_Cat"
VAL = ["EV_baseline", "EV_after_neutral", "EV_after_neg"]
ARO = ["ER_baseline", "ER_after_neutral", "ER_after_neg"]


def mixed_anova(d, cols):
    """Type III mixed ANOVA via effect-coded GLM (handles 72 vs 68)."""
    Y = d[cols].to_numpy(float)
    n, J = Y.shape
    dfe, dw = n - 2, J - 1

    g = np.where(d["Age"].to_numpy() == "younger", 1.0, -1.0)
    X = np.column_stack([np.ones(n), g])
    XtX_inv = np.linalg.inv(X.T @ X)
    B = XtX_inv @ X.T @ Y
    E = Y.T @ Y - B.T @ (X.T @ X) @ B                    # error SSCP

    C = np.linalg.qr(np.column_stack([np.ones(J), np.eye(J)[:, :dw]]))[0][:, 1:]
    Estar = C.T @ E @ C
    Sigma = Estar / dfe
    ss_err, df_err = np.trace(Estar), dfe * dw
    eps = (np.trace(Sigma) ** 2) / (dw * np.trace(Sigma @ Sigma))  # GG

    out = {"eps_GG": eps}
    for name, row in [("Condition", 0), ("Condition x Age", 1)]:
        L = np.zeros((1, 2))
        L[0, row] = 1.0
        d_hat = L @ B @ C
        ss = float(np.trace(d_hat.T @ np.linalg.inv(L @ XtX_inv @ L.T) @ d_hat))
        F = (ss / dw) / (ss_err / df_err)
        out[name] = dict(F=F, df1=dw, df2=df_err,
                         p=stats.f.sf(F, dw, df_err),
                         p_GG=stats.f.sf(F, dw * eps, df_err * eps),
                         np2=ss / (ss + ss_err))
    return out


def paired(d, a, b):
    diff = d[a].to_numpy(float) - d[b].to_numpy(float)
    t, p = stats.ttest_rel(d[a], d[b])
    return t, p, diff.mean() / diff.std(ddof=1)


def main():
    path = next((a for a in sys.argv[1:]
                 if a.lower().endswith(".xlsx") and os.path.isfile(a)),
                DATA_PATH)
    df = pd.read_excel(path)
    df["Age"] = (df[AGE_COL].astype(str).str.strip().str.lower()
                 .map({"young": "younger", "younger": "younger",
                       "old": "older", "older": "older"}))

    for cols, name in [(VAL, "VALENCE"), (ARO, "AROUSAL")]:
        d = df[["Age"] + cols].dropna()
        n = d["Age"].value_counts()
        print(f"\n== {name} (N = {len(d)}: younger {n['younger']}, "
              f"older {n['older']}) ==")

        res = mixed_anova(d, cols)
        for eff in ("Condition", "Condition x Age"):
            r = res[eff]
            print(f"{eff}: F({r['df1']}, {r['df2']}) = {r['F']:.2f}, "
                  f"p = {r['p']:.3g}, np2 = {r['np2']:.3f} "
                  f"(GG eps = {res['eps_GG']:.2f}, p_GG = {r['p_GG']:.3g})")

        for a, b, lab in [(cols[2], cols[0], "Negative vs Baseline"),
                          (cols[2], cols[1], "Negative vs Neutral"),
                          (cols[0], cols[1], "Baseline vs Neutral")]:
            t, p, dz = paired(d, a, b)
            print(f"{lab}: t({len(d) - 1}) = {t:.2f}, p = {p:.3g}, "
                  f"dz = {dz:.2f}")

    d = df[["EV_after_neg", "ER_after_neg"]].dropna()
    r, p = stats.pearsonr(d["EV_after_neg"], d["ER_after_neg"])
    print(f"\nValence-arousal correlation (After Negative): "
          f"r = {r:.2f}, p = {p:.3g}, N = {len(d)}")


if __name__ == "__main__":
    main()